In [1]:
import os
import json
import ast
import operator
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Securely load API key
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
assert api_key, "GOOGLE_API_KEY not set — check your .env file."

# Initialize the Gemini client
client = genai.Client(api_key=api_key)
print("Gemini client ready.")

# 2. Load the orders database
try:
    with open("orders.json", "r") as f:
        orders_db = json.load(f)
    print(f"Successfully loaded {len(orders_db)} orders.")
except FileNotFoundError:
    print("Error: orders.json not found in the current directory.")

Gemini client ready.
Successfully loaded 4 orders.


In [2]:
def lookup_order(order_id: str) -> dict:
    """Looks up an order by its ID and returns its details."""
    if not isinstance(order_id, str) or not order_id.strip():
        return {"error": "Invalid order_id format. Must be a non-empty string."}
    
    order = orders_db.get(order_id.strip().upper())
    if order:
        return order
    return {"error": f"Order '{order_id}' not found in the database."}

def calculate(expression: str) -> dict:
    """Evaluates a basic arithmetic expression safely."""
    if not isinstance(expression, str) or not expression.strip():
        return {"error": "Invalid expression format. Must be a string."}
    
    allowed_operators = {
        ast.Add: operator.add, 
        ast.Sub: operator.sub, 
        ast.Mult: operator.mul, 
        ast.Div: operator.truediv
    }

    def safe_eval_node(node):
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return node.value
            raise TypeError("Only numbers are allowed.")
        elif isinstance(node, ast.BinOp):
            return allowed_operators[type(node.op)](safe_eval_node(node.left), safe_eval_node(node.right))
        elif isinstance(node, ast.UnaryOp):
            if isinstance(node.op, ast.USub):
                return -safe_eval_node(node.operand)
            elif isinstance(node.op, ast.UAdd):
                return safe_eval_node(node.operand)
        raise TypeError(f"Unsupported operation or node type: {type(node)}")

    try:
        tree = ast.parse(expression.strip(), mode='eval').body
        result = safe_eval_node(tree)
        return {"result": result}
    except Exception as e:
        return {"error": f"Failed to calculate expression '{expression}': {str(e)}"}

print("Tools defined.")

Tools defined.


In [3]:
lookup_order_schema = types.FunctionDeclaration(
    name="lookup_order",
    description="Looks up an order by its ID and returns its details, including item name, price, purchase date, and warranty length in months.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "order_id": types.Schema(
                type=types.Type.STRING, 
                description="The exact order ID to look up, e.g., 'A1001'."
            ),
        },
        required=["order_id"]
    )
)

calculate_schema = types.FunctionDeclaration(
    name="calculate",
    description="Evaluates a basic mathematical expression. Use this for all arithmetic to ensure accurate multiplication, addition, subtraction, or division.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "expression": types.Schema(
                type=types.Type.STRING, 
                description="The arithmetic expression to evaluate, e.g., '1200 * 3'."
            ),
        },
        required=["expression"]
    )
)

# Package them up for the API
tools = [types.Tool(function_declarations=[lookup_order_schema, calculate_schema])]
print("Schemas packaged into 'tools'.")

Schemas packaged into 'tools'.


In [4]:
def run_agent_loop(prompt: str, messages: list, step_limit: int = 5):
    print(f"\nUser: {prompt}\n" + "="*60)
    
    # APPEND 1: Add the user's new prompt to memory
    messages.append(
        types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    )
    
    loop_count = 0
    
    # THE STEP LIMIT: A real bound
    while loop_count < step_limit:
        loop_count += 1
        print(f"\n--- Loop Iteration {loop_count} ---")
        
        # Call the model with the ENTIRE memory array
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=messages,
            config=types.GenerateContentConfig(
                tools=tools,
                temperature=0.0
            )
        )
        
        # APPEND 2: Add the model's exact reply to memory (whether it's a tool request or text)
        if response.candidates and response.candidates[0].content:
            messages.append(response.candidates[0].content)
        else:
            print("❌ Error: No content returned from model.")
            break
            
        # CONTROL FLOW A: The model requested a tool
        if response.function_calls:
            function_responses = []
            
            for tool_call in response.function_calls:
                tool_name = tool_call.name
                args = tool_call.args if tool_call.args else {}
                
                print(f"🔄 [Model Request]: call {tool_name}({args})")
                
                # Execute the tool
                if tool_name == "lookup_order":
                    result = lookup_order(args.get("order_id", ""))
                elif tool_name == "calculate":
                    result = calculate(args.get("expression", ""))
                else:
                    result = {"error": f"Unknown tool: {tool_name}"}
                    
                print(f"✅ [System Execution]: returned {result}")
                
                # Package the result for the model
                function_responses.append(
                    types.Part.from_function_response(
                        name=tool_name,
                        response=result
                    )
                )
            
            # APPEND 3: Add the tool execution results to memory so the model can read them
            messages.append(
                types.Content(role="user", parts=function_responses)
            )
            
            # Loop continues automatically here to send the updated memory back to the model
            
        # CONTROL FLOW B: The model provided a final text answer
        elif response.text:
            print(f"🤖 Final Answer:\n{response.text}")
            break
            
        else:
            print("❌ Error: Unexpected response format.")
            break
            
    # THE LIMIT BREACH: If the loop finishes without breaking, we hit the ceiling
    if loop_count == step_limit:
        print(f"\n⚠️ Loop aborted: couldn't finish in time (hit limit of {step_limit} steps).")

In [5]:
# Initialize our literal short-term memory
conversation_memory = []

# Run Turn 1
run_agent_loop(
    prompt="What did order A1001 cost?", 
    messages=conversation_memory, 
    step_limit=5
)


User: What did order A1001 cost?

--- Loop Iteration 1 ---
🔄 [Model Request]: call lookup_order({'order_id': 'A1001'})
✅ [System Execution]: returned {'item': 'laptop', 'price': 1200, 'purchased': '2026-05-20', 'warranty_months': 12}

--- Loop Iteration 2 ---
🤖 Final Answer:
The cost of order A1001 was $1200.


In [6]:
# Pass the exact same list from Turn 1. 
# Do NOT re-initialize conversation_memory!
run_agent_loop(
    prompt="And what about three of them?", 
    messages=conversation_memory, 
    step_limit=5
)


User: And what about three of them?

--- Loop Iteration 1 ---
🔄 [Model Request]: call calculate({'expression': '1200 * 3'})
✅ [System Execution]: returned {'result': 3600}

--- Loop Iteration 2 ---
🤖 Final Answer:
Three of them would cost $3600.


In [7]:
print("\n--- Testing the Step Limit ---")
# Use a fresh memory list so it doesn't get confused
limit_test_memory = []

# Give it a multi-tool task but only 1 allowed step
run_agent_loop(
    prompt="Lookup order A1002, then lookup A1003, and calculate their combined price.",
    messages=limit_test_memory,
    step_limit=1
)


--- Testing the Step Limit ---

User: Lookup order A1002, then lookup A1003, and calculate their combined price.

--- Loop Iteration 1 ---
🔄 [Model Request]: call lookup_order({'order_id': 'A1002'})
✅ [System Execution]: returned {'item': 'wireless mouse', 'price': 30, 'purchased': '2026-06-01', 'warranty_months': 6}

⚠️ Loop aborted: couldn't finish in time (hit limit of 1 steps).


In [ ]:
# The MCP Stretch (Conceptual Demo)
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def mcp_stretch_demo():
    # 1. Define the connection to an existing MCP server (e.g., fetch)
    server_params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-fetch"]
    )
    
    # 2. Establish the connection
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # DIFFERENCE 1: Standardized tool discovery. 
            # We don't write the schema; we ask the server what tools it has.
            mcp_tools = await session.list_tools()
            print("Discovered MCP Tools:", [t.name for t in mcp_tools.tools])
            
            # In a real loop, we would convert 'mcp_tools' to Gemini types here,
            # send them to the model, and if the model requested "fetch", we execute:
            
            # DIFFERENCE 2: Standardized execution.
            # result = await session.call_tool("fetch", {"url": "https://google.com"})
            # print(result)

await mcp_stretch_demo()

### Note on MCP Execution Error (`UnsupportedOperation: fileno`)

The MCP stretch goal code above is structurally correct and demonstrates the proper way to connect to a standardized tool server. However, executing it directly in this cell throws an `UnsupportedOperation: fileno` error. 

**Why this happens:**
This is a known environment clash, not a code failure. The `stdio_client` attempts to spawn the MCP server as a background child process. To capture the server's output, it tries to hook into the operating system's standard file descriptors (`fileno()`). Because Jupyter Notebooks replace standard terminal streams with simulated streams to print outputs into these cells, the underlying Windows `subprocess` module panics when it cannot find a true OS-level file descriptor.

**How to run it:**
To see the tool discovery and execution work, this specific function must be run in a standard terminal environment (e.g., saved as a `.py` script and executed via the command line) where real OS file streams are available.